## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


확률적 경사 하강법(SGD, Stochastic Gradient Descent)를 사용할 수 있다.

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

과적합이 발생하고 있는 것이다.

과적합을 해결하기 위해서 조기종료 (Early Stopping)를 이용하거나 L1, L2 등의 규제를 이용한다. 또는 모델을 단순화하거나 데이터를 늘린다.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

높은 편향이 문제이다.

규제 하이퍼파라미터 $\alpha$를 줄여야 한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

- 평범한 선형 회귀 대신 릿지 회귀 -> 과적합을 줄이기 위해 규제를 적용한다.
- 릿지 회귀 대신 라쏘 회귀 -> 불필요한 특성을 제거하고 특성 선택(feature selection)을 하기 위해 라쏘 회귀를 사용한다.
- 라쏘 회귀 대신 엘라스틱넷 -> 특성들이 강하게 상관되어 있거나 특성이 많을 때 라쏘가 불안정해질 수 있으므로 L1+L2 규제를 함께 사용해 안정적으로 학습하기 위해 엘라스틱넷을 사용한다.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [ ]:
import numpy as np
from sklearn.datasets import load_iris

# 데이터 로드
iris = load_iris()
X = iris["data"]
y = iris["target"]

# bias 추가
X = np.c_[np.ones((len(X), 1)), X]

# train / validation 분할
np.random.seed(42)
indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

train_size = int(len(X) * 0.8)
X_train = X[:train_size]
X_val = X[train_size:]
y_train = y[:train_size]
y_val = y[train_size:]

# 원핫 인코딩
def one_hot(y):
  n_classes = np.max(y) + 1
  m = len(y)
  one_hot = np.zeros((m, n_classes))
  one_hot[np.arange(m), y] = 1
  return one_hot

y_train_onehot = one_hot(y_train)
y_val_onehot = one_hot(y_val)

# 소프트맥스 함수
def softmax(logits):
  exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
  return exps / np.sum(exps, axis=1, keepdims=True)

# 초기 설정
eta = 0.1
n_iterations = 5000
m = len(X_train)
n_inputs = X_train.shape[1]
n_outputs = 3

Theta = np.random.randn(n_inputs, n_outputs)

best_loss = np.inf
patience = 20
count = 0

# 배치 경사 하강법
for iteration in range(n_iterations):
  logits = X_train.dot(Theta)
  Y_proba = softmax(logits)

  loss = -np.mean(np.sum(y_train_onehot * np.log(Y_proba + 1e-7), axis=1))

  gradients = (1/m) * X_train.T.dot(Y_proba - y_train_onehot)
  Theta -= eta * gradients

  # validation loss
  val_logits = X_val.dot(Theta)
  val_proba = softmax(val_logits)
  val_loss = -np.mean(np.sum(y_val_onehot * np.log(val_proba + 1e-7), axis=1))

  # Early Stopping
  if val_loss < best_loss:
    best_loss = val_loss
    best_theta = Theta.copy()
    count = 0
  else:
    count += 1
    if count >= patience:
      print("Early stopping at iteration:", iteration)
      break

Theta = best_theta

# 예측
logits = X_val.dot(Theta)
y_pred = np.argmax(softmax(logits), axis=1)

accuracy = np.mean(y_pred == y_val)
print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.9666666666666667
